In [2]:
# ============================================================
# NOTEBOOK — Per-Class Accuracy, Full Cross-Dataset Run
# NIH ChestX-ray14 (age + sex) + Fed-ISIC2019 (institutional)
# CLIP ViT-L/14 + ResNet-50
# GPU T4, Internet ON. ~25 min.
# ============================================================

import subprocess
subprocess.run(["pip", "install", "transformers", "torch", "torchvision",
                "scikit-learn", "pandas", "numpy", "datasets", "-q"], check=False)

import torch
import numpy as np, pandas as pd, os, json, warnings
from PIL import Image
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torchvision.models as tvm
import torchvision.transforms as tvt
from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_BOOTSTRAP  = 500
np.random.seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ============================================================
# MODELS
# ============================================================
print("Loading ResNet-50...")
resnet = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device).eval()
resnet_transform = tvt.Compose([
    tvt.Resize(256), tvt.CenterCrop(224), tvt.ToTensor(),
    tvt.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("ResNet-50 loaded.")

print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
print("CLIP loaded.")

# ============================================================
# FEATURE EXTRACTION
# ============================================================
@torch.no_grad()
def get_clip_features(images, batch_size=32):
    if len(images) == 0:
        return np.empty((0, 768), dtype=np.float32)
    all_feats = []
    for i in range(0, len(images), batch_size):
        pv = clip_processor(images=images[i:i+batch_size],
                            return_tensors="pt")["pixel_values"].to(device)
        f  = clip_model.vision_model(pixel_values=pv).pooler_output
        f  = clip_model.visual_projection(f).float()
        f  = f / f.norm(dim=-1, keepdim=True)
        all_feats.append(f.cpu().numpy())
        if i % 320 == 0: print(f"  CLIP {i}/{len(images)}...")
    return np.vstack(all_feats)

@torch.no_grad()
def get_resnet_features(images, batch_size=32):
    if len(images) == 0:
        return np.empty((0, 2048), dtype=np.float32)
    all_feats = []
    for i in range(0, len(images), batch_size):
        t = torch.stack([resnet_transform(img)
                         for img in images[i:i+batch_size]]).to(device)
        f = resnet(t).float()
        f = f / f.norm(dim=-1, keepdim=True)
        all_feats.append(f.cpu().numpy())
        if i % 320 == 0: print(f"  ResNet {i}/{len(images)}...")
    return np.vstack(all_feats)

def load_images_from_paths(paths):
    imgs, valid_idx = [], []
    for i, p in enumerate(paths):
        try:
            imgs.append(Image.open(p).convert('RGB').resize((224, 224)))
            valid_idx.append(i)
        except: pass
    return imgs, valid_idx

# ============================================================
# EVALUATION
# ============================================================
def wilson_ci(p, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    denom  = 1 + z**2/n
    centre = (p + z**2/(2*n)) / denom
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return (float(centre - margin), float(centre + margin))

def evaluate(train_f, train_y, test_f, test_y, label_names, name):
    if len(train_f) == 0 or len(test_f) == 0:
        print(f"  [{name}] SKIPPED — empty split"); return None
    if len(np.unique(train_y)) < 2:
        print(f"  [{name}] SKIPPED — only 1 class in train"); return None

    clf   = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
    clf.fit(train_f, train_y)
    probs = clf.predict_proba(test_f)
    preds = clf.predict(test_f)

    # AUC
    try:
        auc = roc_auc_score(test_y, probs, multi_class='ovr', average='macro')
    except:
        auc = roc_auc_score(test_y, probs[:, 1])

    # Bootstrap CI
    scores = []
    for _ in range(N_BOOTSTRAP):
        idx = np.random.choice(len(test_y), len(test_y), replace=True)
        try:
            scores.append(roc_auc_score(test_y[idx], probs[idx],
                                        multi_class='ovr', average='macro'))
        except: pass
    ci_low, ci_high = (np.percentile(scores, [2.5, 97.5])
                       if scores else (auc-.02, auc+.02))

    # Per-class accuracy
    per_class = {}
    for i, cls in enumerate(label_names):
        cls = str(cls)
        mask = (test_y == i)
        n = int(mask.sum())
        if n > 0:
            p = float(accuracy_score(test_y[mask], preds[mask]))
            per_class[cls] = {'acc': p, 'n': n, 'ci': wilson_ci(p, n)}

    print(f"\n  [{name}]")
    print(f"  AUC={auc:.4f} ({ci_low:.4f}-{ci_high:.4f})")
    for cls, pc in per_class.items():
        print(f"    {cls}: {pc['acc']:.3f} "
              f"(95% CI: {pc['ci'][0]:.3f}-{pc['ci'][1]:.3f}, n={pc['n']})")

    return {'name': name, 'auc': float(auc),
            'ci_low': float(ci_low), 'ci_high': float(ci_high),
            'per_class': per_class}

# ============================================================
# SECTION 1: NIH ChestX-ray14
# ============================================================
print("\n" + "="*60)
print("SECTION 1: NIH ChestX-ray14")
print("="*60)

nih_path = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
df_nih   = pd.read_csv(os.path.join(nih_path, 'Data_Entry_2017.csv'))
df_nih   = df_nih[(df_nih['Patient Age'] > 0) & (df_nih['Patient Age'] < 120)]
df_nih['has_finding'] = (df_nih['Finding Labels'] != 'No Finding').astype(int)

print("Building image index...")
img_index = {}
for d in os.listdir(nih_path):
    if d.startswith('images_'):
        img_dir = os.path.join(nih_path, d, 'images')
        if os.path.exists(img_dir):
            for f in os.listdir(img_dir):
                img_index[f] = os.path.join(img_dir, f)
print(f"Indexed {len(img_index):,} images")

df_nih['image_path'] = df_nih['Image Index'].map(img_index)
df_nih = df_nih[df_nih['image_path'].notna()].copy()
nih_labels = ['no_finding', 'has_finding']

# ── Age split ─────────────────────────────────────────────────
print("\n--- Age Split (train: young <40, test: older >60) ---")
young = df_nih[df_nih['Patient Age'] < 40].sample(2000, random_state=RANDOM_STATE)
older = df_nih[df_nih['Patient Age'] > 60].sample(2000, random_state=RANDOM_STATE)

young_imgs, yv = load_images_from_paths(young['image_path'].tolist())
older_imgs,  ov = load_images_from_paths(older['image_path'].tolist())
young_y = young['has_finding'].values[yv]
older_y = older['has_finding'].values[ov]
print(f"Young: {len(young_imgs)} (path rate {young_y.mean():.3f}), "
      f"Older: {len(older_imgs)} (path rate {older_y.mean():.3f})")

young_clip   = get_clip_features(young_imgs)
older_clip   = get_clip_features(older_imgs)
young_resnet = get_resnet_features(young_imgs)
older_resnet = get_resnet_features(older_imgs)

nih_age = {}
print("\n-- CLIP Age Split --")
nih_age['clip']   = evaluate(young_clip,   young_y, older_clip,   older_y, nih_labels, "CLIP Age-Aware")
print("\n-- ResNet-50 Age Split --")
nih_age['resnet'] = evaluate(young_resnet, young_y, older_resnet, older_y, nih_labels, "ResNet-50 Age-Aware")

# ── Sex split (honest negative) ───────────────────────────────
print("\n--- Sex Split / Honest Negative (train: male, test: female) ---")
male_df   = df_nih[df_nih['Patient Gender'] == 'M'].sample(1000, random_state=RANDOM_STATE)
female_df = df_nih[df_nih['Patient Gender'] == 'F'].sample(1000, random_state=RANDOM_STATE)

male_imgs,   mv = load_images_from_paths(male_df['image_path'].tolist())
female_imgs, fv = load_images_from_paths(female_df['image_path'].tolist())
male_y   = male_df['has_finding'].values[mv]
female_y = female_df['has_finding'].values[fv]
print(f"Male: {len(male_imgs)} (path rate {male_y.mean():.3f}), "
      f"Female: {len(female_imgs)} (path rate {female_y.mean():.3f})")

male_clip     = get_clip_features(male_imgs)
female_clip   = get_clip_features(female_imgs)
male_resnet   = get_resnet_features(male_imgs)
female_resnet = get_resnet_features(female_imgs)

nih_sex = {}
print("\n-- CLIP Sex Split --")
nih_sex['clip']   = evaluate(male_clip,   male_y, female_clip,   female_y, nih_labels, "CLIP Sex-Aware")
print("\n-- ResNet-50 Sex Split --")
nih_sex['resnet'] = evaluate(male_resnet, male_y, female_resnet, female_y, nih_labels, "ResNet-50 Sex-Aware")

# ============================================================
# SECTION 2: Fed-ISIC2019
# ============================================================
print("\n" + "="*60)
print("SECTION 2: Fed-ISIC2019")
print("="*60)

isic_ds = load_dataset("flwrlabs/fed-isic2019")
print(f"Splits: {list(isic_ds.keys())}")

def hf_to_df(split):
    rows = []
    for ex in isic_ds[split]:
        rows.append({
            'image':     ex['image'],
            'label':     ex['label'],
            'center_id': ex.get('center_id', ex.get('institution_id', 0)),
        })
    return pd.DataFrame(rows)

isic_df = hf_to_df('train')
if 'test' in isic_ds:
    isic_df = pd.concat([isic_df, hf_to_df('test')], ignore_index=True)

print(f"Total: {len(isic_df)}")
print(f"Institutions: {sorted(isic_df['center_id'].unique())}")
print(f"Label dist:\n{isic_df['label'].value_counts()}")

isic_le = LabelEncoder()
isic_df['label_enc'] = isic_le.fit_transform(isic_df['label'])
isic_labels = [str(c) for c in isic_le.classes_]

institutions = sorted(isic_df['center_id'].unique())
if len(institutions) > 1:
    test_inst  = institutions[-1]
    train_pool = isic_df[isic_df['center_id'] != test_inst]
    test_pool  = isic_df[isic_df['center_id'] == test_inst]
    split_type = f"institutional (test={test_inst})"
else:
    print("Single institution — falling back to stratified 75/25 split")
    train_pool, test_pool = train_test_split(
        isic_df, test_size=0.25, random_state=RANDOM_STATE,
        stratify=isic_df['label_enc'])
    split_type = "random 75/25 fallback"

train_isic = train_pool.sample(min(3000, len(train_pool)), random_state=RANDOM_STATE)
test_isic  = test_pool.sample(min(1000,  len(test_pool)),  random_state=RANDOM_STATE)
print(f"Split type: {split_type}")
print(f"Train: {len(train_isic)}, Test: {len(test_isic)}")

def extract_hf_images(df_sub):
    imgs, lbls = [], []
    for _, row in df_sub.iterrows():
        try:
            imgs.append(row['image'].convert('RGB').resize((224, 224)))
            lbls.append(int(row['label_enc']))
        except: pass
    return imgs, np.array(lbls)

train_imgs, train_y = extract_hf_images(train_isic)
test_imgs,  test_y  = extract_hf_images(test_isic)
print(f"Train images: {len(train_imgs)}, Test images: {len(test_imgs)}")

train_clip  = get_clip_features(train_imgs)
test_clip   = get_clip_features(test_imgs)
train_resnet = get_resnet_features(train_imgs)
test_resnet  = get_resnet_features(test_imgs)

isic_results = {}
print("\n-- CLIP ISIC --")
isic_results['clip']   = evaluate(train_clip,   train_y, test_clip,   test_y, isic_labels, "CLIP ISIC")
print("\n-- ResNet-50 ISIC --")
isic_results['resnet'] = evaluate(train_resnet, train_y, test_resnet, test_y, isic_labels, "ResNet-50 ISIC")

# ============================================================
# SAVE + PRINT SUMMARY
# ============================================================
all_results = {
    'nih_age_split':      {k: v for k, v in nih_age.items()  if v},
    'nih_sex_split':      {k: v for k, v in nih_sex.items()  if v},
    'isic_institutional': {k: v for k, v in isic_results.items() if v},
}

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
for section, results in all_results.items():
    print(f"\n{section}:")
    for arch, r in results.items():
        if r:
            print(f"  {arch}: AUC={r['auc']:.4f} ({r['ci_low']:.4f}-{r['ci_high']:.4f})")
            for cls, pc in r['per_class'].items():
                print(f"    {cls}: {pc['acc']:.3f} CI=({pc['ci'][0]:.3f}-{pc['ci'][1]:.3f}) n={pc['n']}")

out_path = '/kaggle/working/perclass_crossdataset_final.json'
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"\n✓ Saved to {out_path}")
print("\n✓ Complete. Paste ALL output back to Claude.")

Device: cuda
Loading ResNet-50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 176MB/s] 


ResNet-50 loaded.
Loading CLIP...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded.

SECTION 1: NIH ChestX-ray14
Building image index...
Indexed 112,120 images

--- Age Split (train: young <40, test: older >60) ---
Young: 2000 (path rate 0.422), Older: 2000 (path rate 0.526)
  CLIP 0/2000...
  CLIP 320/2000...
  CLIP 640/2000...
  CLIP 960/2000...
  CLIP 1280/2000...
  CLIP 1600/2000...
  CLIP 1920/2000...
  CLIP 0/2000...
  CLIP 320/2000...
  CLIP 640/2000...
  CLIP 960/2000...
  CLIP 1280/2000...
  CLIP 1600/2000...
  CLIP 1920/2000...
  ResNet 0/2000...
  ResNet 320/2000...
  ResNet 640/2000...
  ResNet 960/2000...
  ResNet 1280/2000...
  ResNet 1600/2000...
  ResNet 1920/2000...
  ResNet 0/2000...
  ResNet 320/2000...
  ResNet 640/2000...
  ResNet 960/2000...
  ResNet 1280/2000...
  ResNet 1600/2000...
  ResNet 1920/2000...

-- CLIP Age Split --

  [CLIP Age-Aware]
  AUC=0.6407 (0.6207-0.6607)
    no_finding: 0.768 (95% CI: 0.740-0.793, n=947)
    has_finding: 0.409 (95% CI: 0.380-0.439, n=1053)

-- ResNet-50 Age Split --

  [ResNet-50 Age-Aware]
  AU